In [1]:
import pandas as pd
#import janitor
import numpy as np
import re, os, sqlite3
from datetime import datetime
import pytz

In [2]:
# Function to cleanup and split time range in the response

def clean_time_range(df, column_name):
    cleaned = [] 
    for i in range(len(df[column_name])):
        if pd.notna(df[column_name][i]) and str(df[column_name][i]).startswith('time_range'):
            t = re.sub(r'[a-zA-Z\s+(\)_:]', '', df[column_name][i])
            t = t.replace(',', ':')
            if re.search(r'^[0-9]:', t): #9,30/12,30
                ttemp = '0' + t
            elif re.search(r':[0-9]$', t): #12,5/12,30
                ttemp = t.replace(':', ':0')
            else:
                ttemp = t
            # tpos = datetime.strptime(str(ttemp), '%H:%M')
            # thm = tpos.strftime('%H:%M')
            thm = ttemp      
        elif pd.notna(df[column_name][i]) and str(df[column_name][i]).startswith(' to'):
            t = re.sub(r'[a-zA-Z\s+(\)_:]', '', df[column_name][i])
            t = t.replace(',', ':')
            if re.search(r'^[0-9]:', t): #9,30/12,30
                ttemp = '0' + t
            elif re.search(r':[0-9]$', t): #12,5/12,30
                ttemp = t.replace(':', ':0')
            else:
                ttemp = t
            thm = ttemp      
        else:
            ttemp = df[column_name][i] 
            thm = ttemp
        cleaned.append(thm)
    return cleaned

In [3]:
#setup path variable and get list of report.csv files

input_path = os.path.expanduser('~/NIMH EMA Data/HBN Input/')
files = os.listdir(os.path.join(input_path, 'EMA_applet_data'))

output_path = os.path.expanduser('~/NIMH EMA Data/HBN Output/')

In [4]:
#Removing .DS_Store files

file_list = np.array([file[0:3] for file in files])
files = np.asarray(files)
files = files[np.where(file_list=="HBN")[0]]

In [5]:
#Read all the report.csv files
report_all = []
for i in range(len(files)):
    temp_df = pd.read_csv(os.path.join(input_path, 'EMA_applet_data', files[i]))
    report_all.append(temp_df)

/var/folders/l0/d7yxt_9s3sv5gmjd8qk19p2h0000gs/T/ipykernel_88985/1705551678.py:4: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(os.path.join(input_path, 'EMA_applet_data', files[i]))


In [6]:
# Concat report.csv to one file and read other input files

dat_full = pd.concat(report_all, ignore_index=True)
dat_full = dat_full.applymap(str)
dat_full.head()

/var/folders/l0/d7yxt_9s3sv5gmjd8qk19p2h0000gs/T/ipykernel_88985/2162791287.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  dat_full = dat_full.applymap(str)


,id,activity_scheduled_time,activity_start_time,activity_end_time,flag,secret_user_id,userId,activity_id,activity_name,activity_flow,item,response,prompt,options,version,rawScore,reviewing_id
0,65466c8c2304487093910260,not scheduled,1699114109078,1699114117468,completed,18673,653c14373c36ce79135b31b8,5f0c3695efb9bd41b2e89b60,Pre-Questionnaire,nan,socialmedia,value: 1,![new/allSocialMedia.png](https://raw.githubus...,"Yes: 1, No: 0",0.0.3,0,nan
1,65466c8c2304487093910260,not scheduled,1699114109078,1699114117468,completed,18673,653c14373c36ce79135b31b8,5f0c3695efb9bd41b2e89b60,Pre-Questionnaire,nan,socialmedia_type,value: 4,What social media accounts do you use?,"Twitter: 1, Instagram: 2, Facebook: 3, Snapcha...",0.0.3,0,nan
2,65466bbb230448709391025a,1699113600000,1699113629548,1699113904073,completed,18673,653c14373c36ce79135b31b8,5f0c3697efb9bd41b2e89b64,EMA Assessment (Mid Day),nan,since_activity_monitor,value: 2,![new/wgt3x.png](https://raw.githubusercontent...,"I did not receive an activity monitor: 2, Yes:...",0.0.3,0,nan
3,65466bbb230448709391025a,1699113600000,1699113629548,1699113904073,completed,18673,653c14373c36ce79135b31b8,5f0c3697efb9bd41b2e89b64,EMA Assessment (Mid Day),nan,now_where,value: 2,Where are you **right now**?,"In my home: 1, in home of relative or friend: ...",0.0.3,0,nan
4,65466bbb230448709391025a,1699113600000,1699113629548,1699113904073,completed,18673,653c14373c36ce79135b31b8,5f0c3697efb9bd41b2e89b64,EMA Assessment (Mid Day),nan,since_where,value: 1,Which places have you been **since the last qu...,"In my home: 1, in home of relative or friend: ...",0.0.3,0,nan


In [7]:
# starting from here is the translation of R code

dat_full['start_Time'] = dat_full['activity_start_time']
dat_full['end_Time'] = dat_full['activity_end_time']
dat_full['schedule_Time'] = dat_full['activity_scheduled_time']

In [8]:
# Cleanup similar to R code

dat_full = dat_full[dat_full['activity_scheduled_time']!="not scheduled"]
dat_full = dat_full.reset_index()
dat_full = dat_full.drop('index',axis=1)
dat_processed = dat_full.groupby(['secret_user_id', 'activity_id', 'activity_scheduled_time'], group_keys=True).apply(lambda x: x.assign(start_Time=x['start_Time'].min(), end_Time=x['end_Time'].max())).reset_index(drop=True)
dat_processed.head(20)

,id,activity_scheduled_time,activity_start_time,activity_end_time,flag,secret_user_id,userId,activity_id,activity_name,activity_flow,item,response,prompt,options,version,rawScore,reviewing_id,start_Time,end_Time,schedule_Time
0,6125165bd28f9adace06f1af,1629795600000,1629820219852,1629820506926,completed,10338,61117f9d66f506a576da45c6,5f0c3696efb9bd41b2e89b62,EMA Assessment (Morning),nan,morning_bedtime,minute: 0 | hour: 22,![new/bedClock.png](https://raw.githubusercont...,nan,0.0.2,0,nan,1629820219852,1629820506926,1629795600000
1,6125165bd28f9adace06f1af,1629795600000,1629820219852,1629820506926,completed,10338,61117f9d66f506a576da45c6,5f0c3696efb9bd41b2e89b62,EMA Assessment (Morning),nan,morning_lights_off,minute: 0 | hour: 23,![new/lampClock.png](https://raw.githubusercon...,nan,0.0.2,0,nan,1629820219852,1629820506926,1629795600000
2,6125165bd28f9adace06f1af,1629795600000,1629820219852,1629820506926,completed,10338,61117f9d66f506a576da45c6,5f0c3696efb9bd41b2e89b62,EMA Assessment (Morning),nan,morning_close_eyes,minute: 55 | hour: 22,![new/closedEyeClock.png](https://raw.githubus...,nan,0.0.2,0,nan,1629820219852,1629820506926,1629795600000
3,6125165bd28f9adace06f1af,1629795600000,1629820219852,1629820506926,completed,10338,61117f9d66f506a576da45c6,5f0c3696efb9bd41b2e89b62,EMA Assessment (Morning),nan,morning_sleep_latency,value: 3,![new/zzzEyeTimer.png](https://raw.githubuserc...,"less than 5 minutes: 1, 5 minutes: 2, 10 minut...",0.0.2,0,nan,1629820219852,1629820506926,1629795600000
4,6125165bd28f9adace06f1af,1629795600000,1629820219852,1629820506926,completed,10338,61117f9d66f506a576da45c6,5f0c3696efb9bd41b2e89b62,EMA Assessment (Morning),nan,morning_wake_number,value: 1,![new/awakeNight.png](https://raw.githubuserco...,"none: 1, once: 2, twice: 3, 3 or more times: 4",0.0.2,0,nan,1629820219852,1629820506926,1629795600000
5,6125165bd28f9adace06f1af,1629795600000,1629820219852,1629820506926,completed,10338,61117f9d66f506a576da45c6,5f0c3696efb9bd41b2e89b62,EMA Assessment (Morning),nan,morning_waketime,minute: 40 | hour: 8,![new/alarmClock.png](https://raw.githubuserco...,nan,0.0.2,0,nan,1629820219852,1629820506926,1629795600000
6,6125165bd28f9adace06f1af,1629795600000,1629820219852,1629820506926,completed,10338,61117f9d66f506a576da45c6,5f0c3696efb9bd41b2e89b62,EMA Assessment (Morning),nan,morning_outbed,minute: 20 | hour: 9,![new/outBedClock.png](https://raw.githubuserc...,nan,0.0.2,0,nan,1629820219852,1629820506926,1629795600000
7,6125165bd28f9adace06f1af,1629795600000,1629820219852,1629820506926,completed,10338,61117f9d66f506a576da45c6,5f0c3696efb9bd41b2e89b62,EMA Assessment (Morning),nan,morning_sleep_quantity,value: 17,![new/zzzBedTimer.png](https://raw.githubuserc...,"less than 30 minutes: 1, 30 minutes: 2, 1 hour...",0.0.2,0,nan,1629820219852,1629820506926,1629795600000
8,6125165bd28f9adace06f1af,1629795600000,1629820219852,1629820506926,completed,10338,61117f9d66f506a576da45c6,5f0c3696efb9bd41b2e89b62,EMA Assessment (Morning),nan,morning_sleep_quality,value: 7,How would you rate the **quality** of your sleep?,"1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7",0.0.2,0,nan,1629820219852,1629820506926,1629795600000
9,6125165bd28f9adace06f1af,1629795600000,1629820219852,1629820506926,completed,10338,61117f9d66f506a576da45c6,5f0c3696efb9bd41b2e89b62,EMA Assessment (Morning),nan,morning_sleep_refreshed,value: 7,How **refreshed** did you feel when you woke up?,"1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7",0.0.2,0,nan,1629820219852,1629820506926,1629795600000


In [9]:
# added extra columns than what exists in R cleanup code

dat_subset = dat_processed.iloc[:, [0, 1,5,6,7,8,10,11,13,17,18,19,14]]
dat_subset.head()

,id,activity_scheduled_time,secret_user_id,userId,activity_id,activity_name,item,response,options,start_Time,end_Time,schedule_Time,version
0,6125165bd28f9adace06f1af,1629795600000,10338,61117f9d66f506a576da45c6,5f0c3696efb9bd41b2e89b62,EMA Assessment (Morning),morning_bedtime,minute: 0 | hour: 22,nan,1629820219852,1629820506926,1629795600000,0.0.2
1,6125165bd28f9adace06f1af,1629795600000,10338,61117f9d66f506a576da45c6,5f0c3696efb9bd41b2e89b62,EMA Assessment (Morning),morning_lights_off,minute: 0 | hour: 23,nan,1629820219852,1629820506926,1629795600000,0.0.2
2,6125165bd28f9adace06f1af,1629795600000,10338,61117f9d66f506a576da45c6,5f0c3696efb9bd41b2e89b62,EMA Assessment (Morning),morning_close_eyes,minute: 55 | hour: 22,nan,1629820219852,1629820506926,1629795600000,0.0.2
3,6125165bd28f9adace06f1af,1629795600000,10338,61117f9d66f506a576da45c6,5f0c3696efb9bd41b2e89b62,EMA Assessment (Morning),morning_sleep_latency,value: 3,"less than 5 minutes: 1, 5 minutes: 2, 10 minut...",1629820219852,1629820506926,1629795600000,0.0.2
4,6125165bd28f9adace06f1af,1629795600000,10338,61117f9d66f506a576da45c6,5f0c3696efb9bd41b2e89b62,EMA Assessment (Morning),morning_wake_number,value: 1,"none: 1, once: 2, twice: 3, 3 or more times: 4",1629820219852,1629820506926,1629795600000,0.0.2


In [10]:
# Cleanup similar to R code

for i in range(len(dat_subset['response'])):  
    if re.search(r'value', dat_subset['response'][i]):
        if re.search(r'value: ', dat_subset['response'][i]):
            r = dat_subset['response'][i].replace("value: ", "")
            dat_subset.loc[i, 'response'] = r
        elif re.search(r'value', dat_subset['response'][i]):
            r = dat_subset['response'][i].replace("value:", 'NA')
            dat_subset.loc[i, 'response'] = r
    elif re.search(r'minute:', dat_subset['response'][i]):
        if re.search(r'hour: [0-9]$', dat_subset['response'][i]): 
            egapp = dat_subset['response'][i]. replace(' | hour: ', ':0')
            if re.search(r'minute: [0-9]:', egapp): 
                egtemp = egapp.replace('minute: ', '0')
            elif re.search(r'minute: [0-9][0-9]', egapp): 
                egtemp = egapp.replace('minute: ', '')
            egpos = datetime.strptime(egtemp, '%M:%H')
            egpos2 = egpos.strftime('%H:%M')
            dat_subset.loc[i, 'response'] = egpos2
        elif re.search(r'hour: [0-9][0-9]', dat_subset['response'][i]):
            egapp = dat_subset['response'][i].replace(' | hour: ', ':')
            if re.search(r'minute: [0-9]:', egapp): 
                egtemp = egapp.replace('minute: ', '0')
            elif re.search(r'minute: [0-9][0-9]', egapp): 
                egtemp = egapp.replace('minute: ', '')
            egpos = datetime.strptime(egtemp, '%M:%H')
            egpos2 = egpos.strftime('%H:%M')
            dat_subset.loc[i, 'response'] = egpos2

/var/folders/l0/d7yxt_9s3sv5gmjd8qk19p2h0000gs/T/ipykernel_88985/478966561.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dat_subset.loc[i, 'response'] = egpos2
/var/folders/l0/d7yxt_9s3sv5gmjd8qk19p2h0000gs/T/ipykernel_88985/478966561.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dat_subset.loc[i, 'response'] = r
/var/folders/l0/d7yxt_9s3sv5gmjd8qk19p2h0000gs/T/ipykernel_88985/478966561.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view

In [11]:
# Combining scores and other formats of responses into one column and selecting required columns

#dat_subset['response2'] = np.where(dat_subset['response_scores'].isna(), dat_subset['response'], dat_subset['response_scores'])
dat_subset = dat_subset[['userId', 'secret_user_id', 'activity_name', 'activity_id', 'item', 'response', 
                         'options', 'start_Time',
                         'end_Time', 'schedule_Time', 'activity_scheduled_time', 'version', 'id']]
dat_subset.head(15)

,userId,secret_user_id,activity_name,activity_id,item,response,options,start_Time,end_Time,schedule_Time,activity_scheduled_time,version,id
0,61117f9d66f506a576da45c6,10338,EMA Assessment (Morning),5f0c3696efb9bd41b2e89b62,morning_bedtime,22:00,nan,1629820219852,1629820506926,1629795600000,1629795600000,0.0.2,6125165bd28f9adace06f1af
1,61117f9d66f506a576da45c6,10338,EMA Assessment (Morning),5f0c3696efb9bd41b2e89b62,morning_lights_off,23:00,nan,1629820219852,1629820506926,1629795600000,1629795600000,0.0.2,6125165bd28f9adace06f1af
2,61117f9d66f506a576da45c6,10338,EMA Assessment (Morning),5f0c3696efb9bd41b2e89b62,morning_close_eyes,22:55,nan,1629820219852,1629820506926,1629795600000,1629795600000,0.0.2,6125165bd28f9adace06f1af
3,61117f9d66f506a576da45c6,10338,EMA Assessment (Morning),5f0c3696efb9bd41b2e89b62,morning_sleep_latency,3,"less than 5 minutes: 1, 5 minutes: 2, 10 minut...",1629820219852,1629820506926,1629795600000,1629795600000,0.0.2,6125165bd28f9adace06f1af
4,61117f9d66f506a576da45c6,10338,EMA Assessment (Morning),5f0c3696efb9bd41b2e89b62,morning_wake_number,1,"none: 1, once: 2, twice: 3, 3 or more times: 4",1629820219852,1629820506926,1629795600000,1629795600000,0.0.2,6125165bd28f9adace06f1af
5,61117f9d66f506a576da45c6,10338,EMA Assessment (Morning),5f0c3696efb9bd41b2e89b62,morning_waketime,08:40,nan,1629820219852,1629820506926,1629795600000,1629795600000,0.0.2,6125165bd28f9adace06f1af
6,61117f9d66f506a576da45c6,10338,EMA Assessment (Morning),5f0c3696efb9bd41b2e89b62,morning_outbed,09:20,nan,1629820219852,1629820506926,1629795600000,1629795600000,0.0.2,6125165bd28f9adace06f1af
7,61117f9d66f506a576da45c6,10338,EMA Assessment (Morning),5f0c3696efb9bd41b2e89b62,morning_sleep_quantity,17,"less than 30 minutes: 1, 30 minutes: 2, 1 hour...",1629820219852,1629820506926,1629795600000,1629795600000,0.0.2,6125165bd28f9adace06f1af
8,61117f9d66f506a576da45c6,10338,EMA Assessment (Morning),5f0c3696efb9bd41b2e89b62,morning_sleep_quality,7,"1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7",1629820219852,1629820506926,1629795600000,1629795600000,0.0.2,6125165bd28f9adace06f1af
9,61117f9d66f506a576da45c6,10338,EMA Assessment (Morning),5f0c3696efb9bd41b2e89b62,morning_sleep_refreshed,7,"1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7",1629820219852,1629820506926,1629795600000,1629795600000,0.0.2,6125165bd28f9adace06f1af


In [12]:
# Date formatting of the Epoch time to human readable timestamp

dat_subset['activity_scheduled_time'] = pd.to_numeric(dat_subset['activity_scheduled_time'], errors='coerce')
dat_subset['activity_scheduled_time'] = pd.to_datetime(dat_subset['activity_scheduled_time'] / 1000, unit='s')
dat_subset['activity_scheduled_date'] = dat_subset['activity_scheduled_time'].dt.date

In [13]:
# Widening data

dat_wide = pd.pivot_table(dat_subset, index=['userId', 'secret_user_id', 'activity_name', 'start_Time', 'end_Time', 'schedule_Time', 'activity_scheduled_time', 'version'], columns='item', values='response', aggfunc='last').reset_index()

In [14]:
dat_wide.head(15)

item,userId,secret_user_id,activity_name,start_Time,end_Time,schedule_Time,activity_scheduled_time,version,FinalScreen,day_cold_cough_flu,...,since_substances,since_substances_cannabis,since_substances_cigarettes,since_substances_other,since_vigorous_activity,since_vigorous_activity_planned,since_where,socialmedia_after_bedtime,socialmedia_duration,socialmedia_fall_asleep
0,5f108e85d7d16bd6c5f5c71c,00000,EMA Assessment (Morning),1652193925158,1652194054178,1652173200000,2022-05-10 09:00:00,0.0.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),1628535586708,1628535935630,1628524800000,2021-08-09 16:00:00,0.0.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,"8,9",NaN,NaN,NaN
2,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),1628547939123,1628548380142,1628539200000,2021-08-09 20:00:00,0.0.2,NaN,NaN,...,NaN,NaN,NaN,NaN,5,1,"1,8",NaN,NaN,NaN
3,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),1628634287106,1628634683022,1628625600000,2021-08-10 20:00:00,0.0.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,"1,8,12,9",NaN,NaN,NaN
4,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),1628707345789,1628707695041,1628697600000,2021-08-11 16:00:00,0.0.2,NaN,NaN,...,NaN,NaN,NaN,NaN,6,1,"1,12",NaN,NaN,NaN
5,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),1628788318315,1628788545083,1628784000000,2021-08-12 16:00:00,0.0.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN
6,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),1628807446725,1628807771765,1628798400000,2021-08-12 20:00:00,0.0.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,"1,8,2",NaN,NaN,NaN
7,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),1629050662975,1629051226692,1629043200000,2021-08-15 16:00:00,0.0.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,"5,8",NaN,NaN,NaN
8,5f721917df32cb2322ba3afc,11916,EMA Assessment (Morning),1628522149433,1628523142807,1628499600000,2021-08-09 09:00:00,0.0.2,NaN,NaN,...,NA,NaN,NaN,NaN,NaN,NaN,"1,9,8,12",1,NaN,0
9,5f721917df32cb2322ba3afc,11916,EMA Assessment (Morning),1628601085657,1628601639657,1628586000000,2021-08-10 09:00:00,0.0.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,0,NaN,NaN


In [15]:
# Using the function from above to cleanup time range and split start and end dates from the response

dat_wide[['since_activity_monitor_time_start', 'since_activity_monitor_time_end']] = dat_wide['since_activity_monitor_time'].str.split("/", expand=True)
dat_wide['since_activity_monitor_time_start'] = clean_time_range(dat_wide, 'since_activity_monitor_time_start')
dat_wide['since_activity_monitor_time_end'] = clean_time_range(dat_wide, 'since_activity_monitor_time_end')

In [16]:
dat_wide_full = dat_wide.applymap(str)

/var/folders/l0/d7yxt_9s3sv5gmjd8qk19p2h0000gs/T/ipykernel_88985/409904353.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  dat_wide_full = dat_wide.applymap(str)


In [17]:
# Epoch to Timestamp

dat_wide_full['start_Time'] = pd.to_numeric(dat_wide_full['start_Time'], errors='coerce')
dat_wide_full['start_Time'] = pd.to_datetime(dat_wide_full['start_Time'] / 1000, unit='s')

In [18]:
# Epoch to Timestamp

dat_wide_full['end_Time'] = pd.to_numeric(dat_wide_full['end_Time'], errors='coerce')
dat_wide_full['end_Time'] = pd.to_datetime(dat_wide_full['end_Time'] / 1000, unit='s')

In [19]:
# Epoch to Timestamp

dat_wide_full['schedule_Time'] = pd.to_numeric(dat_wide_full['schedule_Time'], errors='coerce')
dat_wide_full['schedule_Time'] = pd.to_datetime(dat_wide_full['schedule_Time'] / 1000, unit='s')

In [20]:
# Getting list of secret_user_id for userIds. This step could be excluded in future, once export file and schedule file have same secret_user_id

ID_List = dat_wide_full[['userId', 'secret_user_id']].drop_duplicates()

In [21]:
# Timestamp cleanup

dat_wide_full['schedule_Time'] = pd.to_datetime(dat_wide_full['schedule_Time'])

In [22]:
# Timestamp formatting

dat_wide_full['scheduled_Date'] = dat_wide_full['schedule_Time'].dt.date
dat_wide_full.head(15)

item,userId,secret_user_id,activity_name,start_Time,end_Time,schedule_Time,activity_scheduled_time,version,FinalScreen,day_cold_cough_flu,...,since_substances_other,since_vigorous_activity,since_vigorous_activity_planned,since_where,socialmedia_after_bedtime,socialmedia_duration,socialmedia_fall_asleep,since_activity_monitor_time_start,since_activity_monitor_time_end,scheduled_Date
0,5f108e85d7d16bd6c5f5c71c,00000,EMA Assessment (Morning),2022-05-10 14:45:25.157999872,2022-05-10 14:47:34.177999872,2022-05-10 09:00:00,2022-05-10 09:00:00,0.0.2,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,2022-05-10
1,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),2021-08-09 18:59:46.708000000,2021-08-09 19:05:35.630000128,2021-08-09 16:00:00,2021-08-09 16:00:00,0.0.2,nan,nan,...,nan,nan,nan,"8,9",nan,nan,nan,nan,nan,2021-08-09
2,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),2021-08-09 22:25:39.122999808,2021-08-09 22:33:00.141999872,2021-08-09 20:00:00,2021-08-09 20:00:00,0.0.2,nan,nan,...,nan,5,1,"1,8",nan,nan,nan,16:00,17:10,2021-08-09
3,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),2021-08-10 22:24:47.105999872,2021-08-10 22:31:23.022000128,2021-08-10 20:00:00,2021-08-10 20:00:00,0.0.2,nan,nan,...,nan,nan,nan,"1,8,12,9",nan,nan,nan,08:0,15:00,2021-08-10
4,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),2021-08-11 18:42:25.788999936,2021-08-11 18:48:15.040999936,2021-08-11 16:00:00,2021-08-11 16:00:00,0.0.2,nan,nan,...,nan,6,1,"1,12",nan,nan,nan,22:00,15:00,2021-08-11
5,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),2021-08-12 17:11:58.315000064,2021-08-12 17:15:45.083000064,2021-08-12 16:00:00,2021-08-12 16:00:00,0.0.2,nan,nan,...,nan,nan,nan,1,nan,nan,nan,nan,nan,2021-08-12
6,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),2021-08-12 22:30:46.724999936,2021-08-12 22:36:11.765000192,2021-08-12 20:00:00,2021-08-12 20:00:00,0.0.2,nan,nan,...,nan,nan,nan,"1,8,2",nan,nan,nan,nan,nan,2021-08-12
7,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),2021-08-15 18:04:22.974999808,2021-08-15 18:13:46.692000000,2021-08-15 16:00:00,2021-08-15 16:00:00,0.0.2,nan,nan,...,nan,nan,nan,"5,8",nan,nan,nan,nan,nan,2021-08-15
8,5f721917df32cb2322ba3afc,11916,EMA Assessment (Morning),2021-08-09 15:15:49.433000192,2021-08-09 15:32:22.806999808,2021-08-09 09:00:00,2021-08-09 09:00:00,0.0.2,nan,nan,...,nan,nan,nan,"1,9,8,12",1,nan,0,nan,nan,2021-08-09
9,5f721917df32cb2322ba3afc,11916,EMA Assessment (Morning),2021-08-10 13:11:25.657000192,2021-08-10 13:20:39.657000192,2021-08-10 09:00:00,2021-08-10 09:00:00,0.0.2,nan,nan,...,nan,nan,nan,1,0,nan,nan,23:00,09:0,2021-08-10


In [23]:
egts = dat_wide_full['start_Time']
new_egts = egts.dt.tz_localize('UTC').dt.tz_convert('America/New_York')
dat_wide_full['start_Time'] = new_egts.dt.strftime('%Y-%m-%d %H:%M:%S')

egts2 = dat_wide_full['end_Time']
new_egts2 = egts2.dt.tz_localize('UTC').dt.tz_convert('America/New_York')
dat_wide_full['end_Time'] = new_egts2.dt.strftime('%Y-%m-%d %H:%M:%S')

egts3 = dat_wide_full['schedule_Time']
new_egts3 = egts3.dt.tz_localize('UTC').dt.tz_convert('America/New_York')
dat_wide_full['schedule_Time'] = new_egts3.dt.strftime('%Y-%m-%d %H:%M:%S')

dat_wide_full.head()

item,userId,secret_user_id,activity_name,start_Time,end_Time,schedule_Time,activity_scheduled_time,version,FinalScreen,day_cold_cough_flu,...,since_substances_other,since_vigorous_activity,since_vigorous_activity_planned,since_where,socialmedia_after_bedtime,socialmedia_duration,socialmedia_fall_asleep,since_activity_monitor_time_start,since_activity_monitor_time_end,scheduled_Date
0,5f108e85d7d16bd6c5f5c71c,00000,EMA Assessment (Morning),2022-05-10 10:45:25,2022-05-10 10:47:34,2022-05-10 05:00:00,2022-05-10 09:00:00,0.0.2,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,2022-05-10
1,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),2021-08-09 14:59:46,2021-08-09 15:05:35,2021-08-09 12:00:00,2021-08-09 16:00:00,0.0.2,nan,nan,...,nan,nan,nan,"8,9",nan,nan,nan,nan,nan,2021-08-09
2,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),2021-08-09 18:25:39,2021-08-09 18:33:00,2021-08-09 16:00:00,2021-08-09 20:00:00,0.0.2,nan,nan,...,nan,5,1,"1,8",nan,nan,nan,16:00,17:10,2021-08-09
3,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),2021-08-10 18:24:47,2021-08-10 18:31:23,2021-08-10 16:00:00,2021-08-10 20:00:00,0.0.2,nan,nan,...,nan,nan,nan,"1,8,12,9",nan,nan,nan,08:0,15:00,2021-08-10
4,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),2021-08-11 14:42:25,2021-08-11 14:48:15,2021-08-11 12:00:00,2021-08-11 16:00:00,0.0.2,nan,nan,...,nan,6,1,"1,12",nan,nan,nan,22:00,15:00,2021-08-11


In [24]:
# selecting the required columns and ordering

final = dat_wide_full
final = final[list(final.columns)]
final.head(20)

item,userId,secret_user_id,activity_name,start_Time,end_Time,schedule_Time,activity_scheduled_time,version,FinalScreen,day_cold_cough_flu,...,since_substances_other,since_vigorous_activity,since_vigorous_activity_planned,since_where,socialmedia_after_bedtime,socialmedia_duration,socialmedia_fall_asleep,since_activity_monitor_time_start,since_activity_monitor_time_end,scheduled_Date
0,5f108e85d7d16bd6c5f5c71c,00000,EMA Assessment (Morning),2022-05-10 10:45:25,2022-05-10 10:47:34,2022-05-10 05:00:00,2022-05-10 09:00:00,0.0.2,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,2022-05-10
1,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),2021-08-09 14:59:46,2021-08-09 15:05:35,2021-08-09 12:00:00,2021-08-09 16:00:00,0.0.2,nan,nan,...,nan,nan,nan,"8,9",nan,nan,nan,nan,nan,2021-08-09
2,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),2021-08-09 18:25:39,2021-08-09 18:33:00,2021-08-09 16:00:00,2021-08-09 20:00:00,0.0.2,nan,nan,...,nan,5,1,"1,8",nan,nan,nan,16:00,17:10,2021-08-09
3,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),2021-08-10 18:24:47,2021-08-10 18:31:23,2021-08-10 16:00:00,2021-08-10 20:00:00,0.0.2,nan,nan,...,nan,nan,nan,"1,8,12,9",nan,nan,nan,08:0,15:00,2021-08-10
4,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),2021-08-11 14:42:25,2021-08-11 14:48:15,2021-08-11 12:00:00,2021-08-11 16:00:00,0.0.2,nan,nan,...,nan,6,1,"1,12",nan,nan,nan,22:00,15:00,2021-08-11
5,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),2021-08-12 13:11:58,2021-08-12 13:15:45,2021-08-12 12:00:00,2021-08-12 16:00:00,0.0.2,nan,nan,...,nan,nan,nan,1,nan,nan,nan,nan,nan,2021-08-12
6,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),2021-08-12 18:30:46,2021-08-12 18:36:11,2021-08-12 16:00:00,2021-08-12 20:00:00,0.0.2,nan,nan,...,nan,nan,nan,"1,8,2",nan,nan,nan,nan,nan,2021-08-12
7,5f721917df32cb2322ba3afc,11916,EMA Assessment (Mid Day),2021-08-15 14:04:22,2021-08-15 14:13:46,2021-08-15 12:00:00,2021-08-15 16:00:00,0.0.2,nan,nan,...,nan,nan,nan,"5,8",nan,nan,nan,nan,nan,2021-08-15
8,5f721917df32cb2322ba3afc,11916,EMA Assessment (Morning),2021-08-09 11:15:49,2021-08-09 11:32:22,2021-08-09 05:00:00,2021-08-09 09:00:00,0.0.2,nan,nan,...,nan,nan,nan,"1,9,8,12",1,nan,0,nan,nan,2021-08-09
9,5f721917df32cb2322ba3afc,11916,EMA Assessment (Morning),2021-08-10 09:11:25,2021-08-10 09:20:39,2021-08-10 05:00:00,2021-08-10 09:00:00,0.0.2,nan,nan,...,nan,nan,nan,1,0,nan,nan,23:00,09:0,2021-08-10


In [25]:
# Ensuring consistent NA wording across all columns

final_out = final.copy()
final_out.replace('nan', 'NA', inplace=True)
final_out.fillna('NA', inplace=True)
final_out.replace('NA NA', 'NA', inplace=True)
#final_out.rename(columns = {'userId_x':'user_id'},inplace=True)

In [26]:
# Sorting

final_out = final_out.sort_values(by=['secret_user_id', 'schedule_Time'])

In [27]:
# Included similar to R script to ensure alignment with formatting

final_out = final_out.applymap(str)
final_out = final_out.drop_duplicates()

/var/folders/l0/d7yxt_9s3sv5gmjd8qk19p2h0000gs/T/ipykernel_88985/3812529574.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  final_out = final_out.applymap(str)


In [28]:
# Write the final output file to csv

final_out.to_csv(os.path.join(output_path, 'hbn_ema_output.csv'), index=False, date_format='%Y-%m-%d %H:%M:%S', na_rep='NA')

In [29]:
# checking for duplicates in final file. Ensure this always returns "False"

final_out.duplicated().any()

False